# Bounded-prefix evaluation

This notebook summarizes available bounded-prefix evaluations and preserves unavailable statuses in the coverage export.

In [ ]:
from pathlib import Path
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from notebooks.utils.report_diagrams_utils import (available_metric_rows, normalized_confusion_matrix, report_output_paths, save_report_figure, save_report_table)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
OUTPUTS = report_output_paths(DATA_ROOT)
RUN_ID = "bounded-prefix-classifier-evaluation-window32-2026-09-24"
RESULT_ROOT = DATA_ROOT / "experiment-results/classification/evaluations" / RUN_ID
sns.set_theme(style="whitegrid", context="paper")

In [ ]:
metrics = pd.read_csv(RESULT_ROOT / "prefix_evaluation_metrics.csv")
coverage = pd.read_csv(RESULT_ROOT / "prefix_evaluation_coverage.csv")
save_report_table(metrics, OUTPUTS.tables / "bounded_prefix_metrics_all_statuses.csv")
save_report_table(coverage, OUTPUTS.tables / "bounded_prefix_coverage_all_statuses.csv")
available = metrics.loc[metrics["status"].eq("available")].copy()
available = available_metric_rows(available, "macro_f1")
save_report_table(available, OUTPUTS.tables / "bounded_prefix_metrics_available.csv")

In [ ]:
available["token_bound"] = available["min_tokens"].astype(int).astype(str) + "-" + available["max_tokens"].astype(int).astype(str)
long = available.melt(id_vars=["dataset", "token_bound"], value_vars=["balanced_accuracy", "macro_f1", "roc_auc"], var_name="metric", value_name="score")
figure, axis = plt.subplots(figsize=(12, 6), constrained_layout=True)
sns.lineplot(data=long, x="token_bound", y="score", hue="metric", style="dataset", markers=True, dashes=False, ax=axis)
axis.set(title="Available bounded-prefix performance", xlabel="Prefix token range", ylabel="Score", ylim=(0, 1))
save_report_figure(figure, OUTPUTS.figures / "bounded_prefix_performance")
plt.close(figure)

In [ ]:
confusion = normalized_confusion_matrix(available)
save_report_table(confusion.rename_axis("true_class").reset_index(), OUTPUTS.tables / "bounded_prefix_confusion_matrix_normalized.csv")
figure, axis = plt.subplots(figsize=(5, 4), constrained_layout=True)
sns.heatmap(confusion, annot=True, fmt=".3f", vmin=0, vmax=1, cmap="Purples", ax=axis)
axis.set(title="Normalized available-prefix confusion matrix", xlabel="Predicted class", ylabel="True class")
save_report_figure(figure, OUTPUTS.figures / "bounded_prefix_confusion_matrix_normalized")
plt.close(figure)